# Synthetic Augmentation Comparison

**Purpose**: compare the four synthetic-augmentation probe runs that feed the evaluation chapter.

**Inputs** (GCS JSON from `00_model_training`):
- Sweep `low` preset from `aug_sweep_results.json`
- Ablation `high_vol` and `fatal` from the joint ablation run
- Ablation `baseline` from the separate baseline-only runtime

**Output**: summary table plus four chapter figures saved under `OUTPUT_DIR`.


## 1. Install dependencies
Run this cell to install the packages required for this notebook.


In [ ]:
%pip install -q matplotlib pandas

import matplotlib, pandas as pd, numpy as _np
print(f"matplotlib {matplotlib.__version__} | pandas {pd.__version__} | numpy {_np.__version__}")


## 2. Setup and configuration
Pull the three result JSONs from GCS (or local overrides), then extract the four rows used below.

Override any URI with the matching environment variable if a newer run lands in a different folder.


In [ ]:
import os
import json
import shutil
import subprocess
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ── Paths ────────
# Override with environment variables if the files move.
CKPT_BUCKET = os.getenv("LAP_CKPT_BUCKET", "training-checkpoints-REPLACE_K_NUMBER")
CACHE_DIR = Path(os.getenv("LAP_CACHE_DIR", Path.home() / ".cache" / "laparoscopic_seg" / "aug_compare"))
OUTPUT_DIR = Path(os.getenv("LAP_OUTPUT_DIR", "/tmp/laparoscopic_synth_aug_comparison"))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

SWEEP_JSON = os.getenv(
	"LAP_AUG_SWEEP_JSON",
	f"gs://{CKPT_BUCKET}/experiments/"
	"synthetic-augmentation-sweep_augsweep_Unet_resnet34_bce_lr1e-04_p20/"
	"20260809-153639/aug_sweep_results.json",
)
ABLATION_JSON = os.getenv(
	"LAP_AUG_ABLATION_JSON",
	f"gs://{CKPT_BUCKET}/experiments/"
	"synthetic-augmentation-ablation_augablate_Unet_resnet34_bce_lr1e-04_p20/"
	"20260809-181212/aug_ablation_results.json",
)
BASELINE_JSON = os.getenv(
	"LAP_AUG_BASELINE_JSON",
	f"gs://{CKPT_BUCKET}/experiments/"
	"synthetic-augmentation-ablation-baseline_augablate_Unet_resnet34_bce_lr1e-04_p20/"
	"20260809-191843/aug_ablation_results.json",
)

def _gsutil_available() -> bool:
	return shutil.which("gsutil") is not None

def load_json(uri_or_path: str) -> dict:
	"""Load JSON from a local path or gs:// URI (cached under CACHE_DIR)."""
	if uri_or_path.startswith("gs://"):
		if not _gsutil_available():
			raise FileNotFoundError(
				f"Need gsutil to fetch {uri_or_path}. Install the Google Cloud SDK, "
				"or download the file and set the matching LAP_AUG_*_JSON to a local path."
			)
		parts = uri_or_path.rstrip("/").split("/")
		local = CACHE_DIR / f"{parts[-2]}_{parts[-1]}"
		print(f"Fetching {uri_or_path}")
		subprocess.run(["gsutil", "-q", "cp", uri_or_path, str(local)], check=True)
		path = local
	else:
		path = Path(uri_or_path)
		if not path.is_file():
			raise FileNotFoundError(path)
	with open(path) as f:
		return json.load(f)

def row_from_sweep(doc: dict, preset: str) -> dict:
	for r in doc["runs"]:
		if r.get("preset") == preset:
			return {
				"run_key": preset,
				"label": f"Low (full suite, p={r.get('p', '?')})",
				"source": "sweep",
				"source_run": doc.get("run"),
				"source_run_id": doc.get("run_id"),
				"epochs": doc.get("epochs"),
				"fold": doc.get("fold"),
				"iou_tiny": float(r["iou_tiny"]),
				"iou": float(r["iou"]),
				"notool_fp": float(r["notool_fp"]),
				"recall": float(r.get("recall", float("nan"))),
				"precision_strict": float(r.get("precision_strict", float("nan"))),
				"miss_rate": float(r.get("miss_rate", float("nan"))),
				"iou_tiny_bestthr": float(r.get("iou_tiny_bestthr", float("nan"))),
				"detail": r,
			}
	raise KeyError(f"preset '{preset}' not in sweep runs: {[r.get('preset') for r in doc['runs']]}")

def row_from_ablation(doc: dict, variant: str, label=None) -> dict:
	for r in doc["runs"]:
		if r.get("variant") == variant:
			return {
				"run_key": variant,
				"label": label or r.get("label") or variant,
				"source": "ablation",
				"source_run": doc.get("run"),
				"source_run_id": doc.get("run_id"),
				"epochs": doc.get("epochs"),
				"fold": doc.get("fold"),
				"iou_tiny": float(r["iou_tiny"]),
				"iou": float(r["iou"]),
				"notool_fp": float(r["notool_fp"]),
				"recall": float(r.get("recall", float("nan"))),
				"precision_strict": float(r.get("precision_strict", float("nan"))),
				"miss_rate": float(r.get("miss_rate", float("nan"))),
				"iou_tiny_bestthr": float(r.get("iou_tiny_bestthr", float("nan"))),
				"detail": r,
			}
	raise KeyError(
		f"variant '{variant}' not in ablation runs: {[r.get('variant') for r in doc['runs']]}"
	)

sweep_doc = load_json(SWEEP_JSON)
ablation_doc = load_json(ABLATION_JSON)
try:
	baseline_doc = load_json(BASELINE_JSON)
except Exception as e:
	raise FileNotFoundError(
		"Baseline ablation JSON not found at:\n"
		f"  {BASELINE_JSON}\n"
		f"({type(e).__name__}: {e})\n"
		"That separate runtime currently only has folds.json in GCS. "
		"Re-run / finish the baseline ablation so aug_ablation_results.json uploads, "
		"or set LAP_AUG_BASELINE_JSON to the correct path."
	) from e

ROWS = [
	row_from_ablation(baseline_doc, "baseline", label="Baseline"),
	row_from_ablation(ablation_doc, "high_vol", label="High-vol risk"),
	row_from_sweep(sweep_doc, "low"),
	row_from_ablation(ablation_doc, "fatal", label="Fatal risk"),
]
df = pd.DataFrame(ROWS)

print("=" * 60)
print(f"sweep     : {SWEEP_JSON}")
print(f"ablation  : {ABLATION_JSON}")
print(f"baseline  : {BASELINE_JSON}")
print(f"output    : {OUTPUT_DIR}")
print("=" * 60)
for _, r in df.iterrows():
	print(
		f"{r['run_key']:10s} | tiny {r['iou_tiny']:.4f} | IoU {r['iou']:.4f} | "
		f"ntFP {r['notool_fp']:.0f} | {r['source']} {r['source_run_id']} | {r['epochs']}ep"
	)


## 3. Summary table
Numbers behind the four figures. `notool_fp` is mean false-positive pixels on tool-absent frames (the safety axis).


In [ ]:
summary = df[
	["label", "run_key", "iou_tiny", "iou", "notool_fp", "recall", "precision_strict", "epochs", "source_run_id"]
].copy()
summary = summary.rename(columns={
	"iou": "iou_overall",
	"notool_fp": "notool_fp_mean",
})
display(summary.round(4))

csv_path = OUTPUT_DIR / "synth_aug_comparison.csv"
summary.to_csv(csv_path, index=False)
print(f"saved: {csv_path}")


## 4. Chapter figures

1. **Tiny-tool recovery** - ordered `iou_tiny` bars (hypothesis headline)
2. **Trade-off scatter** - overall IoU vs `iou_tiny`, marker size from `notool_fp`
3. **Collateral damage** - `iou_tiny` beside overall IoU per run
4. **Safety check** - `notool_fp` on empty tissue


In [ ]:
COLOURS = {
	"baseline": "#6c757d",
	"high_vol": "#457b9d",
	"low":      "red",
	"fatal":    "#c45c26",
}

df_ord = df.sort_values("iou_tiny").reset_index(drop=True)
labels_ord = df_ord["label"].tolist()
colours_ord = [COLOURS[k] for k in df_ord["run_key"]]

def _save(fig, name: str):
	path = OUTPUT_DIR / name
	fig.savefig(path, dpi=150, bbox_inches="tight")
	print(f"saved: {path}")

# ── 1. Headline: tiny-tool recovery ───────────────────────
fig, ax = plt.subplots(figsize=(8, 4.5))
x = np.arange(len(df_ord))
bars = ax.bar(x, df_ord["iou_tiny"], color=colours_ord, edgecolor="white", width=0.7)
ax.set_xticks(x)
ax.set_xticklabels(labels_ord, rotation=15, ha="right")
ax.set_ylabel("iou_tiny")
ax.set_title("Tiny-tool recovery across synthetic-augmentation probes")
ax.grid(True, axis="y", alpha=0.35)
for b, v in zip(bars, df_ord["iou_tiny"]):
	ax.text(
		b.get_x() + b.get_width() / 2, v + 0.002, f"{v:.3f}",
		ha="center", va="bottom", fontsize=9,
	)
wi = int(np.argmax(df_ord["iou_tiny"].to_numpy()))
ax.patches[wi].set_linewidth(2.0)
ax.patches[wi].set_edgecolor("black")
fig.tight_layout()
_save(fig, "01_tiny_tool_recovery.png")
plt.show()

# ── 2. Trade-off scatter ──────────────────────────────────
fig, ax = plt.subplots(figsize=(6.5, 5.5))
nt = df["notool_fp"].to_numpy()
sizes = 80 + 220 * (nt - nt.min()) / max(nt.max() - nt.min(), 1.0)
ax.scatter(
	df["iou"], df["iou_tiny"],
	s=sizes, c=[COLOURS[k] for k in df["run_key"]],
	edgecolors="black", linewidths=0.6, zorder=3,
)
for _, r in df.iterrows():
	ax.annotate(
		r["label"], (r["iou"], r["iou_tiny"]),
		textcoords="offset points", xytext=(7, 5), fontsize=9,
	)
ax.set_xlabel("overall IoU (tool-present frames)")
ax.set_ylabel("iou_tiny")
ax.set_title("Trade-off: tiny IoU vs overall IoU\n(marker size ~ notool_fp_mean)")
ax.grid(True, alpha=0.35)
ax.annotate(
	"better ->", xy=(0.98, 0.02), xycoords="axes fraction",
	ha="right", va="bottom", fontsize=8, color="#333333",
)
ax.annotate(
	"better ^", xy=(0.02, 0.98), xycoords="axes fraction",
	ha="left", va="top", fontsize=8, color="#333333",
)
fig.tight_layout()
_save(fig, "02_tradeoff_scatter.png")
plt.show()

# ── 3. Collateral damage: tiny vs overall ─────────────────
fig, ax = plt.subplots(figsize=(8.5, 4.8))
x = np.arange(len(df))
w = 0.36
ax.bar(
	x - w / 2, df["iou_tiny"], width=w, label="iou_tiny",
	color=[COLOURS[k] for k in df["run_key"]], edgecolor="white",
)
ax.bar(
	x + w / 2, df["iou"], width=w, label="iou_overall",
	color=[COLOURS[k] for k in df["run_key"]], edgecolor="white", alpha=0.45,
)
ax.set_xticks(x)
ax.set_xticklabels(df["label"], rotation=15, ha="right")
ax.set_ylabel("IoU")
ax.set_title("Collateral damage check: tiny-tool IoU vs overall IoU")
ax.grid(True, axis="y", alpha=0.35)
ax.legend(frameon=False)
fig.tight_layout()
_save(fig, "03_collateral_damage.png")
plt.show()

# ── 4. Safety check: no-tool FP ───────────────────────────
fig, ax = plt.subplots(figsize=(8, 4.5))
x = np.arange(len(df))
bars = ax.bar(
	x, df["notool_fp"], color=[COLOURS[k] for k in df["run_key"]],
	edgecolor="white", width=0.7,
)
ax.set_xticks(x)
ax.set_xticklabels(df["label"], rotation=15, ha="right")
ax.set_ylabel("notool_fp_mean (px)")
ax.set_title("Safety check: false-positive area on tool-absent frames")
ax.grid(True, axis="y", alpha=0.35)
for b, v in zip(bars, df["notool_fp"]):
	ax.text(
		b.get_x() + b.get_width() / 2, v + max(df["notool_fp"]) * 0.01,
		f"{v:.0f}", ha="center", va="bottom", fontsize=9,
	)
si = int(np.argmin(df["notool_fp"].to_numpy()))
ax.patches[si].set_linewidth(2.0)
ax.patches[si].set_edgecolor("black")
fig.tight_layout()
_save(fig, "04_safety_notool_fp.png")
plt.show()

print(f"\nAll figures in {OUTPUT_DIR}")
